# [County] County Prosecution Analysis

**Source:** [County] prosecution extract
**Study period:** 2017 to 2024 (filing year)

**Agreed criteria (all counties)**
- **Unit:** one row = one case = one docket number per defendant. Dockets with multiple defendants are split into one case per defendant.
- **Offense classification:** each case is classified by its most serious charge (felony over misdemeanor, then statutory class).
- **Outcome:** each case gets its most severe outcome across all charges (Trial Conviction > Plea > CWOF > Diversion > Acquittal > Dismissal > Other > Unknown). Open cases are excluded from outcome metrics.
- **Missing data:** the coverage table comes first. Fields over 20% missing are marked "insufficient data" and their metrics are not reported. "Unknown" is kept as its own category.

## Setup

In [ ]:
import sys, subprocess, os
repo_root = subprocess.run(["git", "rev-parse", "--show-toplevel"],
                           capture_output=True, text=True).stdout.strip()
sys.path.insert(0, repo_root)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from viz_style import (set_style, stacked_pct_bar, save_fig, normalize_outcome,
                       case_outcome, OUTCOME_COLORS, TEXT_SECONDARY)

set_style()

COUNTY = "county_name"        # e.g. "worcester"
YEARS = range(2017, 2025)     # 2017 to 2024
MISSING_THRESHOLD = 0.20      # fields above this are "insufficient data"
MIN_CELL = 30                 # suppress disparity cells smaller than this

FIG_DIR = os.path.join(repo_root, "figures", COUNTY)
os.makedirs(FIG_DIR, exist_ok=True)

## Load Data
Fill in the paths and map your extract's column names onto the shared names.
If your county has a single file, point both `FILED_PATH` and `DISPOSED_PATH` at it.

In [ ]:
FILED_PATH = "PATH/TO/filed_file"        # all filed charges (includes open cases)
DISPOSED_PATH = "PATH/TO/disposed_file"  # disposed charges

def read_any(path):
    if path.endswith(".parquet"):
        return pd.read_parquet(path)
    if path.endswith((".xlsx", ".xls")):
        return pd.read_excel(path)
    return pd.read_csv(path, low_memory=False)

# Shared name : your column name. Set to None if your county doesn't have it.
COLS = {
    "docket_number":    "DOCKET_COL",
    "defendant_id":     "DEFENDANT_ID_COL",
    "charge_count":     "CHARGE_NUMBER_COL",
    "filing_date":      "FILING_DATE_COL",
    "disposition_date": "DISPOSITION_DATE_COL",
    "disposition":      "DISPOSITION_COL",       # detailed disposition, mapped to buckets below
    "charge_level":     "FELONY_MISD_COL",       # felony / misdemeanor
    "offense_category": "OFFENSE_CATEGORY_COL",
    "court":            "COURT_COL",
    "race":             "RACE_COL",
    "sex":              "SEX_COL",
    "sentence_type":    "SENTENCE_TYPE_COL",
}

def standardize(raw):
    rename = {v: k for k, v in COLS.items() if v is not None and v in raw.columns}
    out = raw.rename(columns=rename)
    for k in COLS:
        if k not in out.columns:
            out[k] = np.nan
    out["case_key"] = out["docket_number"].astype(str) + "_" + out["defendant_id"].astype(str)
    out["filing_date"] = pd.to_datetime(out["filing_date"], errors="coerce")
    out["disposition_date"] = pd.to_datetime(out["disposition_date"], errors="coerce")
    out["filing_year"] = out["filing_date"].dt.year
    return out

filed = standardize(read_any(FILED_PATH))
disposed = standardize(read_any(DISPOSED_PATH))

missing_cols = [k for k, v in COLS.items() if v is None or v not in read_any(FILED_PATH).columns]
print("Not found in extract:", missing_cols)
print(f"Filed: {len(filed):,} charges, {filed['case_key'].nunique():,} cases")
print(f"Disposed: {len(disposed):,} charges, {disposed['case_key'].nunique():,} cases")

## Map Dispositions and Exclusions
Run the value counts, then map every disposition onto the shared buckets:
**Trial Conviction, Plea, CWOF, Diversion, Acquittal, Dismissal, Other, Unknown**, plus **Pending** for open charges.
Nolle prosequi counts as Dismissal. Unmapped values fall into Other, so check that the Other share is small.

In [ ]:
disposed["disposition"].value_counts(dropna=False).head(40)

In [ ]:
COUNTY_OUTCOMES = {
    # "raw disposition value": "Shared bucket",
    # "dismissed": "Dismissal",
    # "nolle prosequi": "Dismissal",
    # "guilty": "Plea",
    # "jury trial guilty": "Trial Conviction",
    # "CWOF": "CWOF",
    # "pretrial probation": "Diversion",
    # "jury trial not guilty": "Acquittal",
    # "no disposition": "Pending",
}

# Non-criminal outcomes (e.g. civil infractions) and offense categories the client excluded
EXCLUDE_DISPOSITIONS = []   # raw disposition values for civil / non-criminal matters
EXCLUDE_OFFENSES = []       # e.g. ["Motor Vehicle"]

disposed["outcome"] = disposed["disposition"].replace(COUNTY_OUTCOMES).map(normalize_outcome)
disposed = disposed[~disposed["disposition"].isin(EXCLUDE_DISPOSITIONS)]
disposed = disposed[~disposed["offense_category"].isin(EXCLUDE_OFFENSES)]
filed = filed[~filed["offense_category"].isin(EXCLUDE_OFFENSES)]

print(disposed["outcome"].value_counts(dropna=False))
print(f"Share in Other: {(disposed['outcome'] == 'Other').mean():.1%}")

## 0. Data Coverage
Share missing and years available for each key field. Fields above 20% missing are marked **insufficient data** and the metrics that depend on them are not reported.

In [ ]:
COVERAGE_FIELDS = ["docket_number", "defendant_id", "filing_date", "disposition_date",
                   "disposition", "charge_level", "offense_category", "court",
                   "race", "sex", "sentence_type"]

def years_available(d, field):
    yrs = d.loc[d[field].notna(), "filing_year"].dropna()
    return f"{int(yrs.min())}-{int(yrs.max())}" if len(yrs) else "none"

coverage = pd.DataFrame({
    "pct_missing": [disposed[f].isna().mean() * 100 for f in COVERAGE_FIELDS],
    "years_available": [years_available(disposed, f) for f in COVERAGE_FIELDS],
}, index=COVERAGE_FIELDS).round(1)
coverage["status"] = np.where(coverage["pct_missing"] > MISSING_THRESHOLD * 100,
                              "insufficient data", "ok")

INSUFFICIENT = set(coverage.index[coverage["status"] == "insufficient data"])
print("Insufficient:", INSUFFICIENT or "none")
coverage

**Notes:** [Describe any coverage gaps, e.g. a field only recorded after a certain year.]

## Build Case Table
One row per case (docket × defendant), restricted to filing years 2017 to 2024.
- **Most serious charge:** felony over misdemeanor, then lowest charge number as a tiebreak. Replace the tiebreak with statutory class if your extract has it.
- **Outcome:** most severe outcome across charges. Cases with any pending charge are marked Pending and excluded from outcome metrics.

In [ ]:
LEVEL_RANK = {"Felony": 1, "Misdemeanor": 2}  # map your charge_level values here; others rank last

def build_cases(d, with_outcome):
    d = d[d["filing_year"].isin(YEARS)].copy()
    d["level_rank"] = d["charge_level"].map(LEVEL_RANK).fillna(9)
    d["charge_count_num"] = pd.to_numeric(d["charge_count"], errors="coerce").fillna(999)
    lead = (d.sort_values(["case_key", "level_rank", "charge_count_num"])
              .groupby("case_key").first())
    out = pd.DataFrame({
        "filing_year": lead["filing_year"],
        "filing_date": d.groupby("case_key")["filing_date"].min(),
        "charge_level": lead["charge_level"].fillna("Unknown"),
        "offense_category": lead["offense_category"].fillna("Unknown"),
        "court": lead["court"].fillna("Unknown"),
        "race": lead["race"].fillna("Unknown"),
        "sex": lead["sex"].fillna("Unknown"),
        "defendant_id": lead["defendant_id"],
        "n_charges": d.groupby("case_key").size(),
    })
    if with_outcome:
        out["outcome"] = d.groupby("case_key")["outcome"].agg(case_outcome)
        out["disposition_date"] = d.groupby("case_key")["disposition_date"].max()
    return out.reset_index()

filed_cases = build_cases(filed, with_outcome=False)
all_disposed_cases = build_cases(disposed, with_outcome=True)
n_pending = (all_disposed_cases["outcome"] == "Pending").sum()
cases = all_disposed_cases[all_disposed_cases["outcome"] != "Pending"].copy()

print(f"Filed cases 2017-2024: {len(filed_cases):,}")
print(f"Disposed cases 2017-2024: {len(cases):,} ({n_pending:,} pending excluded)")

## 1. Volume and Trend
**Question:** How many cases were filed each year, and how has that changed since 2017?
**Metric:** case count per year, % change year over year and vs. 2017.

In [ ]:
vol = filed_cases.groupby("filing_year").size().rename("cases").to_frame()
vol["pct_change_yoy"] = (vol["cases"].pct_change() * 100).round(1)
vol["pct_change_vs_2017"] = ((vol["cases"] / vol["cases"].iloc[0] - 1) * 100).round(1)

fig, ax = plt.subplots()
ax.bar(vol.index, vol["cases"], color=OUTCOME_COLORS["Trial Conviction"], width=0.7)
for x, n in zip(vol.index, vol["cases"]):
    ax.text(x, n, f"{n:,}", ha="center", va="bottom", fontsize=12)
ax.set_xticks(list(vol.index))
ax.set_ylabel("Cases filed")
ax.set_xlabel("Filing year")
ax.set_title(f"{COUNTY.title()}: cases filed per year")
save_fig(fig, f"{FIG_DIR}/q1_volume.png")
plt.show()
vol

**How this was calculated**
- **Unit:** cases (docket × defendant)
- **Source:** filed extract, so open cases are counted
- **Date field:** filing date (earliest charge in the case)
- **Filters:** filing years 2017 to 2024; [exclusions]

## 2. Per-Capita Rate
**Question:** How many cases are filed per 10,000 residents?
**Metric:** cases / annual county population × 10,000. Use annual ACS or PEP estimates, not a flat 2020 Census figure.

In [ ]:
from county_charts import plot_per_capita

POPULATION = {  # annual county population estimates (Census PEP), July 1
    2017: None, 2018: None, 2019: None, 2020: None,
    2021: None, 2022: None, 2023: None, 2024: None,
}
per_cap = plot_per_capita(vol["cases"].rename(index=int), POPULATION, COUNTY, f"{FIG_DIR}/q2_per_capita.png")
per_cap

**How this was calculated**
- **Numerator:** cases filed per year (from Q1)
- **Denominator:** [population source and vintage]

## 3. Case Outcomes
**Question:** Of charged cases, what share end in dismissal, diversion, plea, trial conviction, or acquittal?
**Metric:** % of disposed criminal cases by most severe outcome, overall and by year. CWOF is reported separately.

In [ ]:
overall = cases["outcome"].value_counts().rename("cases").to_frame()
overall["pct"] = (overall["cases"] / overall["cases"].sum() * 100).round(1)
display(overall)

counts = cases.groupby("filing_year")["outcome"].value_counts().unstack(fill_value=0)
ax = stacked_pct_bar(counts, unit="cases",
                     title=f"{COUNTY.title()}: case outcomes by filing year",
                     xlabel="Filing year")
save_fig(ax.figure, f"{FIG_DIR}/q3_outcomes_by_year.png")
plt.show()

**How this was calculated**
- **Unit:** disposed criminal cases (docket × defendant), pending cases excluded
- **Outcome rule:** most severe outcome across the case's charges
- **Date field:** filing year
- **Mapping:** [list which raw dispositions went into each bucket]

**Note:** Cases filed in 2023 to 2024 that were still open at extraction are missing from the disposed extract, so recent years lean toward fast-resolving cases.

## 4. Sentencing
**Question:** What share of convictions result in incarceration, probation, or fines/other?
**Metric:** % of convicted cases (Trial Conviction or Plea) by most severe sentence type. The missing share is reported next to every rate.

In [ ]:
SENTENCE_RANK = ["Incarceration", "Probation", "Fines/Other", "Unknown"]
SENTENCE_COLORS = {"Incarceration": "#2a78d6", "Probation": "#eb6834",
                   "Fines/Other": "#1baf7a", "Unknown": "#c9c8c2"}

def classify_sentence(text):
    """Parse free-text sentence type. Adjust keywords to your county's values."""
    if pd.isna(text) or str(text).strip() == "":
        return "Unknown"
    t = str(text).upper()
    if any(k in t for k in ["JAIL", "HOC", "HOUSE OF CORRECTION", "PRISON", "INCARC", "COMMITTED"]):
        return "Incarceration"
    if "PROBATION" in t:
        return "Probation"
    return "Fines/Other"

if "sentence_type" in INSUFFICIENT:
    print("Sentence type: insufficient data")
else:
    conv = disposed[disposed["outcome"].isin(["Trial Conviction", "Plea"])
                    & disposed["filing_year"].isin(YEARS)].copy()
    conv["sentence"] = conv["sentence_type"].map(classify_sentence)
    rank = {s: i for i, s in enumerate(SENTENCE_RANK)}
    case_sent = (conv.assign(r=conv["sentence"].map(rank))
                     .sort_values("r").groupby("case_key")
                     .agg(sentence=("sentence", "first"), filing_year=("filing_year", "first")))

    summary = case_sent["sentence"].value_counts().reindex(SENTENCE_RANK, fill_value=0).rename("cases").to_frame()
    summary["pct"] = (summary["cases"] / summary["cases"].sum() * 100).round(1)
    print(f"Missing sentence type: {summary.loc['Unknown', 'pct']}% of convicted cases")
    display(summary)

    by_year = case_sent.groupby("filing_year")["sentence"].value_counts().unstack(fill_value=0)
    by_year = by_year.reindex(columns=SENTENCE_RANK, fill_value=0)
    pct = by_year.div(by_year.sum(axis=1), axis=0) * 100
    fig, ax = plt.subplots()
    bottom = np.zeros(len(pct))
    for s in SENTENCE_RANK:
        ax.bar(pct.index, pct[s], bottom=bottom, color=SENTENCE_COLORS[s], label=s,
               width=0.7, edgecolor="#fcfcfb", linewidth=2)
        for x, v, b in zip(pct.index, pct[s], bottom):
            if v >= 6:
                ax.text(x, b + v / 2, f"{v:.0f}%", ha="center", va="center", fontsize=12)
        bottom += pct[s].values
    for x, n in zip(by_year.index, by_year.sum(axis=1)):
        ax.text(x, 101.5, f"n={n:,}", ha="center", va="bottom", fontsize=12, color=TEXT_SECONDARY)
    ax.set_ylim(0, 108); ax.set_xticks(list(pct.index))
    ax.set_ylabel("% of convicted cases"); ax.set_xlabel("Filing year")
    h, l = ax.get_legend_handles_labels()
    ax.legend(h[::-1], l[::-1], loc="upper left", bbox_to_anchor=(1.01, 1))
    ax.set_title(f"{COUNTY.title()}: sentence type for convicted cases", pad=24)
    save_fig(fig, f"{FIG_DIR}/q4_sentencing.png")
    plt.show()

**How this was calculated**
- **Unit:** convicted cases (Trial Conviction or Plea; CWOF is not a conviction)
- **Sentence rule:** most severe sentence across the case's convicted charges (Incarceration > Probation > Fines/Other)
- **Parsing:** [keywords used to classify sentence text]
- **Missing:** [share with no sentence type]

## 5. Pretrial Detention and Bail
**Question:** What share of defendants are held pretrial, and how often is cash bail set?
**Metric:** % released on personal recognizance, % with cash bail set, % held without bail, % of cash-bail cases unable to post, median bail amount.
Drop out-of-range bail dates before picking the first arraignment.

In [ ]:
from county_charts import plot_bail

BAIL_PATH = None  # path to bail file, or None if your county has no bail data

BAIL_COLS = {  # shared name : your column name (None if your county doesn't have it)
    "docket_number": "DOCKET_COL",
    "defendant_id":  "DEFENDANT_ID_COL",
    "bail_date":     "BAIL_DATE_COL",
    "bail_status":   "BAIL_STATUS_COL",     # raw status, mapped below
    "cash_amount":   "CASH_BAIL_AMOUNT_COL",
    "posted":        None,                  # True/False column if your county records posting
}
BAIL_STATUS_MAP = {  # raw status : PR / Cash Bail / Held Without Bail (unmapped -> Unknown)
    # "personal recognizance": "PR",
}
# If there's no posted column, list the raw statuses that mean "bail set, still held"
NOT_POSTED_STATUSES = []

if BAIL_PATH is None:
    print("No bail data for this county: insufficient data")
else:
    bail = read_any(BAIL_PATH).rename(columns={v: k for k, v in BAIL_COLS.items() if v})
    bail["case_key"] = bail["docket_number"].astype(str) + "_" + bail["defendant_id"].astype(str)
    bail["bail_date"] = pd.to_datetime(bail["bail_date"], errors="coerce")
    bail = bail[bail["bail_date"].dt.year.between(2017, 2024)]          # drop out-of-range dates
    first = bail.sort_values("bail_date").groupby("case_key").first()   # first arraignment
    first = first[first.index.isin(filed_cases["case_key"])]

    if BAIL_COLS["posted"]:
        unable = ~first["posted"].astype("boolean")
    elif NOT_POSTED_STATUSES:
        unable = first["bail_status"].isin(NOT_POSTED_STATUSES)
    else:
        unable = pd.Series(np.nan, index=first.index)

    fy = filed_cases.set_index("case_key")["filing_year"]
    bail_std = pd.DataFrame({
        "case_key": first.index,
        "year": first.index.map(fy).astype(int),
        "status": first["bail_status"].map(BAIL_STATUS_MAP).fillna("Unknown").values,
        "unable_to_post": unable.values,
        "amount": first["cash_amount"].values,
    })
    q5_table, q5_by_year = plot_bail(bail_std, COUNTY, f"{FIG_DIR}/q5_bail_status.png",
                                     post_path=f"{FIG_DIR}/q5_unable_to_post.png",
                                     n_filed=len(filed_cases))
    display(q5_table)

**How this was calculated**
- **Unit:** cases with a bail record; first arraignment per case
- **Coverage:** [share of filed cases with any bail record]
- **Unable to post:** [posted column / statuses counted as bail set but still held / not recorded]
- **Filters:** bail dates outside 2017 to 2024 dropped
- **Chart year:** filing year of the case

## 6. Time to Disposition
**Question:** What is the median time from filing to disposition, and does it differ by case type?
**Metric:** median and IQR of days, by offense category and court. Pending cases and negative intervals are excluded.

In [ ]:
t = cases.copy()
t["days"] = (t["disposition_date"] - t["filing_date"]).dt.days
n_negative = (t["days"] < 0).sum()
t = t[t["days"] >= 0]
print(f"{len(t):,} cases used; {n_negative:,} negative intervals excluded")

def iqr_table(group):
    g = t.groupby(group)["days"]
    out = pd.DataFrame({"cases": g.size(), "median_days": g.median(),
                        "q1": g.quantile(0.25), "q3": g.quantile(0.75)})
    return out[out["cases"] >= MIN_CELL].sort_values("median_days", ascending=False).round(0)

by_offense = iqr_table("offense_category")
by_court = iqr_table("court")
print(f"Overall median: {t['days'].median():.0f} days (IQR {t['days'].quantile(.25):.0f}-{t['days'].quantile(.75):.0f})")

top = by_offense.head(15).iloc[::-1]
fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(top.index, top["median_days"], color=OUTCOME_COLORS["Trial Conviction"])
ax.errorbar(top["median_days"], range(len(top)),
            xerr=[top["median_days"] - top["q1"], top["q3"] - top["median_days"]],
            fmt="none", ecolor=TEXT_SECONDARY, capsize=3)
for y, (m, n) in enumerate(zip(top["median_days"], top["cases"])):
    ax.text(m + 3, y + 0.3, f"{m:.0f} (n={n:,})", fontsize=11)
ax.set_xlabel("Median days from filing to disposition (bars show IQR)")
ax.grid(axis="x"); ax.grid(axis="y", visible=False)
ax.set_title(f"{COUNTY.title()}: time to disposition by offense category")
save_fig(fig, f"{FIG_DIR}/q6_time_by_offense.png")
plt.show()
display(by_court)

**How this was calculated**
- **Unit:** disposed cases
- **Interval:** earliest filing date to latest disposition date in the case
- **Filters:** pending cases and negative intervals excluded; categories with fewer than 30 cases hidden

## 7. Disparities by Race and Sex
**Question:** Do dismissal and incarceration rates differ by race/ethnicity and sex?
**Metric:** rate per group, plus rate ratios vs. White and vs. Male. Unknown race is kept as its own group. Groups under 30 cases are suppressed.

In [ ]:
REF = {"race": "White", "sex": "Male"}  # adjust to your county's labels

def disparity(df, group, flag, label):
    g = df.groupby(group)[flag].agg(["size", "mean"]).rename(columns={"size": "cases", "mean": "rate"})
    g = g[g["cases"] >= MIN_CELL]
    ref = REF[group]
    g["rate_pct"] = (g["rate"] * 100).round(1)
    g["ratio_vs_" + ref] = (g["rate"] / g.loc[ref, "rate"]).round(2) if ref in g.index else np.nan
    return g.drop(columns="rate").assign(metric=label)

cases["dismissed"] = cases["outcome"] == "Dismissal"
results = []
for group in ["race", "sex"]:
    if group in INSUFFICIENT:
        print(f"{group}: insufficient data"); continue
    results.append(disparity(cases, group, "dismissed", "dismissal rate"))
    if "case_sent" in globals():
        inc = case_sent.join(cases.set_index("case_key")[[group]], how="inner")
        inc["incarcerated"] = inc["sentence"] == "Incarceration"
        results.append(disparity(inc, group, "incarcerated", "incarceration rate (convicted cases)"))
disp = pd.concat(results) if results else pd.DataFrame()
disp

In [ ]:
# Dismissal rate by race (chart)
if "race" not in INSUFFICIENT:
    r = disp[(disp["metric"] == "dismissal rate") & disp.index.isin(cases["race"].unique())]
    r = r.sort_values("rate_pct")
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.barh(r.index, r["rate_pct"], color=OUTCOME_COLORS["Dismissal"])
    for y, (v, n) in enumerate(zip(r["rate_pct"], r["cases"])):
        ax.text(v + 0.5, y, f"{v:.1f}% (n={n:,})", va="center", fontsize=12)
    ax.set_xlabel("% of disposed cases dismissed")
    ax.grid(axis="x"); ax.grid(axis="y", visible=False)
    ax.set_title(f"{COUNTY.title()}: dismissal rate by defendant race")
    save_fig(fig, f"{FIG_DIR}/q7_dismissal_by_race.png")
    plt.show()

**How this was calculated**
- **Unit:** disposed cases (dismissal); convicted cases (incarceration)
- **Rate ratio:** group rate / reference group rate (White for race, Male for sex)
- **Filters:** groups under 30 cases suppressed; Unknown kept as its own group

**Note:** Declination (cases the prosecutor declined to file) is [measurable / not measurable] in this extract.

## 8. Disparities with Controls
**Question:** Do racial gaps remain within the same offense category and prior-record level?
**Metric:** dismissal rate by race within offense category × prior-record cells. Cells under 30 cases are suppressed.

**Prior record:** number of earlier cases for the same defendant filed in the **3 years** before this case (same lookback in every county), bucketed 0 / 1-2 / 3+.

In [ ]:
LOOKBACK_YEARS = 3

if "defendant_id" in missing_cols:
    print("No defendant ID: prior record not measurable, insufficient data")
elif "race" in INSUFFICIENT:
    print("Race: insufficient data")
else:
    # Use all filed cases (any year) so cases early in the window still see prior history
    hist = (filed.groupby("case_key").agg(defendant_id=("defendant_id", "first"),
                                          filing_date=("filing_date", "min"))
                 .dropna().sort_values("filing_date"))

    def prior_count(g):
        dates = g["filing_date"].values
        start = dates - np.timedelta64(365 * LOOKBACK_YEARS, "D")
        return pd.Series(np.searchsorted(dates, dates, side="left")
                         - np.searchsorted(dates, start, side="left"), index=g.index)

    hist["prior_cases"] = hist.groupby("defendant_id", group_keys=False).apply(prior_count)
    hist["prior_record"] = pd.cut(hist["prior_cases"], [-1, 0, 2, np.inf], labels=["0", "1-2", "3+"])

    c = cases.merge(hist[["prior_record"]], left_on="case_key", right_index=True, how="left")
    cell = (c.groupby(["offense_category", "prior_record", "race"], observed=True)["dismissed"]
              .agg(cases="size", rate="mean").reset_index())
    cell = cell[cell["cases"] >= MIN_CELL]
    cell["rate_pct"] = (cell["rate"] * 100).round(1)

    wide = cell.pivot_table(index=["offense_category", "prior_record"], columns="race",
                            values="rate_pct", observed=True)
    if REF["race"] in wide.columns:
        for col in wide.columns:
            if col != REF["race"]:
                wide[f"{col} ratio"] = (wide[col] / wide[REF["race"]]).round(2)
    display(wide)

**How this was calculated**
- **Unit:** disposed cases
- **Prior record:** earlier cases for the same defendant ID within 3 years before filing
- **Cells:** offense category × prior record × race; cells under 30 cases suppressed
- **Ratio:** group dismissal rate / White dismissal rate within the same cell

**Limitation:** Prior record only counts cases in this county's extract, not statewide history.

## Questions for the Client
- [e.g., Is a statewide prior-record field available?]
- [e.g., Can declined referrals be provided?]